# Notebook 04 — Training Loop

**Project:** AV-XAI (Explainable Multimodal Deepfake Detection)
**Goal:** Fine-tune the full `AVXAIModel` on the 30K LAV-DF subset. Save best checkpoint by validation balanced accuracy. Log everything needed for the paper.

## Hyperparameters baked in

| Parameter | Value | Why |
|---|---|---|
| Backbone LR | 1e-5 | Standard for full-FT of large pretrained transformers |
| Head LR | 5e-4 | Heads have no pretraining → need higher LR |
| Weight decay | 0.01 (zero for bias & LN) | AdamW best practice |
| Scheduler | Cosine + 500 warmup steps | |
| Max epochs | 8 | |
| Early stop patience | 3 | Plateau-tolerant for cosine schedule |
| Micro-batch | 4 | A6000 fits this comfortably with bf16 |
| Grad accumulation | 8 | Effective batch = 32 |
| Mixed precision | bf16 autocast | |
| Grad clip | 1.0 | Stable for large transformer fine-tune |
| Label smoothing | 0.1 | Robustness to noisy deepfake labels |
| Audio mixup α | 0.2 | Regularizes WavLM (subtle audio artifacts) |
| Contrastive τ | Learnable, init 0.07 | Better than fixed (CLIP-style) |
| Contrastive weight | 0.3 | Per paper spec |

## What gets saved

```
checkpoints/
├── best.pt          ← best by val balanced accuracy (loaded by nb06)
└── last.pt          ← latest epoch (for resumption)

results/
├── tables/
│   └── training_log.csv     ← per-epoch metrics
└── logs/
    └── train_TIMESTAMP.log  ← stdout dump
```

## Notebook structure

- **Phase A** — `AVXAIDataset` (DataLoader over preprocessed .pt files)
- **Phase B** — Audio mixup function
- **Phase C** — Build model + learnable temperature
- **Phase D** — Optimizer with 4 param groups (backbone-decay / backbone-no-decay / head-decay / head-no-decay)
- **Phase E** — Cosine warmup scheduler
- **Phase F** — `train_step` and `validate` functions
- **Phase G** — Smoke test on 50 train + 20 val samples (verifies loop works end-to-end)
- **Phase H** — Full training run (~6 hours, resumable)
- **Phase I** — Save best checkpoint summary

## Estimated runtime

8 epochs × ~45 min = **~6 hours on A6000**. Don't run it without being able to leave the lab machine alone — checkpoint after every epoch means a crash mid-epoch loses at most ~45 minutes.

In [ ]:
# --- Imports and paths ---
import os, sys, json, time, math, random, csv
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

# --- Paths ---
PROJECT_ROOT  = Path(r"C:\Users\Komal-Sch\Desktop\AV-XAI")
DATA_ROOT     = PROJECT_ROOT / "data"
PREPROC_ROOT  = DATA_ROOT / "preprocessed"
SPLITS_DIR    = DATA_ROOT / "splits"
CKPT_DIR      = PROJECT_ROOT / "checkpoints"
RESULTS_DIR   = PROJECT_ROOT / "results"
TABLES_DIR    = RESULTS_DIR / "tables"
LOGS_DIR      = RESULTS_DIR / "logs"

for d in (CKPT_DIR, RESULTS_DIR, TABLES_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Make models/ importable ---
sys.path.insert(0, str(PROJECT_ROOT))
from models.avxai_model import AVXAIModel

# --- Reproducibility ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Training requires CUDA. Check your env."
print(f"Device: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

# --- Master config ---
CFG = {
    "lr_backbone":       1e-5,
    "lr_head":           5e-4,
    "weight_decay":      0.01,
    "warmup_steps":      500,
    "max_epochs":        8,
    "patience":          3,
    "micro_batch":       4,
    "grad_accum":        8,            # effective batch = 32
    "grad_clip":         1.0,
    "label_smoothing":   0.1,
    "audio_mixup_alpha": 0.2,
    "audio_mixup_prob":  0.5,          # apply mixup to 50% of batches
    "tau_init":          0.07,
    "tau_min":           0.01,
    "tau_max":           0.5,
    "contrastive_weight": 0.3,
    "aux_loss_weight":   0.3,          # weight on per-modality auxiliary CE
    "num_workers":       4,
    "log_every_steps":   50,
    "save_dir":          str(CKPT_DIR),
}
print("\nConfig:")
for k, v in CFG.items():
    print(f"  {k:<22}: {v}")

## Phase A — `AVXAIDataset` (DataLoader)

Each sample: load one `.pt` file from `data/preprocessed/{split}/`, return faces (uint8), audio (float32), and labels.

The model's forward pass handles normalization, so we keep faces as uint8 here — saves bandwidth on the DataLoader → GPU transfer.

In [ ]:
class AVXAIDataset(Dataset):
    def __init__(self, manifest_csv: Path, preproc_root: Path):
        self.df = pd.read_csv(manifest_csv)
        self.preproc_root = Path(preproc_root)
        # Pre-resolve filepaths (avoid pandas in __getitem__ for speed)
        self.paths = []
        self.labels_4c = []
        self.labels_bin = []
        for _, row in self.df.iterrows():
            stem = Path(row["file"]).stem
            self.paths.append(self.preproc_root / row["split"] / f"{stem}.pt")
            self.labels_4c.append(int(row["label_4class"]))
            self.labels_bin.append(int(row["label_binary"]))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        entry = torch.load(self.paths[idx], weights_only=False)
        return {
            "faces":         entry["faces"],
            "audio":         entry["audio"],
            "label_4class":  self.labels_4c[idx],
            "label_binary":  self.labels_bin[idx],
        }


def collate_fn(batch):
    return {
        "faces":         torch.stack([b["faces"] for b in batch], dim=0),
        "audio":         torch.stack([b["audio"] for b in batch], dim=0),
        "label_4class":  torch.tensor([b["label_4class"] for b in batch], dtype=torch.long),
        "label_binary":  torch.tensor([b["label_binary"] for b in batch], dtype=torch.long),
    }


# --- Build datasets ---
train_ds = AVXAIDataset(SPLITS_DIR / "train.csv", PREPROC_ROOT)
val_ds   = AVXAIDataset(SPLITS_DIR / "val.csv",   PREPROC_ROOT)
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}")

# --- Smoke test: load 1 sample ---
sample = train_ds[0]
print(f"\nSingle sample:")
print(f"  faces : {tuple(sample['faces'].shape)} {sample['faces'].dtype}")
print(f"  audio : {tuple(sample['audio'].shape)} {sample['audio'].dtype}")
print(f"  label_4class: {sample['label_4class']}")
print(f"  label_binary: {sample['label_binary']}")

# --- Smoke test: load a batch ---
test_loader = DataLoader(train_ds, batch_size=CFG["micro_batch"], shuffle=True,
                          num_workers=0, collate_fn=collate_fn)
batch = next(iter(test_loader))
print(f"\nBatch (micro_batch={CFG['micro_batch']}):")
for k, v in batch.items():
    print(f"  {k:<14}: {tuple(v.shape)}  {v.dtype}")

## Phase B — Audio mixup

Sample λ ~ Beta(α, α), mix two audio waveforms in the batch, and return the mixing coefficient + permutation so the training loop can compute a mixed loss.

We apply mixup with probability `audio_mixup_prob` (50%) so half the batches see clean audio — preserves some clean supervision signal.

**Important:** mixup is applied only to audio, NOT to faces. Face crops can't be mixed semantically (you can't blend two faces and call it a real or fake — the resulting mix is meaningless).

In [ ]:
def audio_mixup(audio: torch.Tensor, labels: torch.Tensor, alpha: float = 0.2):
    """Mix audio waveforms within a batch.

    Returns:
        audio_mixed : mixed waveform
        labels_a    : original labels (primary)
        labels_b    : labels of mixed partners
        lam         : mixing coefficient (scalar in [0, 1])
    """
    B = audio.size(0)
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(B, device=audio.device)
    audio_mixed = lam * audio + (1 - lam) * audio[perm]
    labels_a = labels
    labels_b = labels[perm]
    return audio_mixed, labels_a, labels_b, lam


def mixed_ce(logits: torch.Tensor, labels_a: torch.Tensor, labels_b: torch.Tensor,
             lam: float, label_smoothing: float = 0.1):
    """Mixed cross-entropy: lam·CE(logits, a) + (1-lam)·CE(logits, b)."""
    ce_a = F.cross_entropy(logits, labels_a, label_smoothing=label_smoothing)
    ce_b = F.cross_entropy(logits, labels_b, label_smoothing=label_smoothing)
    return lam * ce_a + (1 - lam) * ce_b


# Smoke test
dummy_audio  = torch.randn(4, 160000)
dummy_labels = torch.tensor([0, 1, 2, 3])
mixed, la, lb, lam = audio_mixup(dummy_audio, dummy_labels, alpha=0.2)
print(f"Mixed audio shape : {tuple(mixed.shape)}")
print(f"lam               : {lam:.4f}")
print(f"labels_a          : {la.tolist()}")
print(f"labels_b          : {lb.tolist()}")

## Phase C — Build model + learnable temperature

Build `AVXAIModel`. Add `log_tau` as a learnable parameter on the top level — initialized to `log(0.07)`, clamped to `[log(0.01), log(0.5)]` during forward.

The temperature shows up only in the contrastive loss; everything else uses the unmodified model.

In [ ]:
print("Building AVXAIModel (loads pretrained weights — first run downloads ~1.5 GB)...")
torch.cuda.empty_cache()

model = AVXAIModel(use_gradient_checkpointing=True).to(device)

# --- Learnable contrastive temperature ---
log_tau = nn.Parameter(torch.log(torch.tensor(CFG["tau_init"])).to(device))
log_tau_min = math.log(CFG["tau_min"])
log_tau_max = math.log(CFG["tau_max"])

def current_tau():
    return log_tau.detach().clamp(log_tau_min, log_tau_max).exp().item()

total_params = sum(p.numel() for p in model.parameters()) + 1
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad) + 1
print(f"Total params    : {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable/1e6:.1f}M")
print(f"Initial τ       : {current_tau():.4f}")

## Phase D — Optimizer with 4 param groups

AdamW correctness requires excluding biases and LayerNorm parameters from weight decay. Combined with our backbone-vs-head differential LR, we end up with **4 param groups**:

| Group | LR | Weight Decay |
|---|---|---|
| Backbone weights (ViViT + WavLM) | 1e-5 | 0.01 |
| Backbone bias + LayerNorm | 1e-5 | 0.0 |
| Head weights (consistency + fusion + classifiers) | 5e-4 | 0.01 |
| Head bias + LayerNorm | 5e-4 | 0.0 |
| `log_tau` (learnable temperature) | 5e-4 | 0.0 |

In [ ]:
def is_no_decay_param(name: str) -> bool:
    return name.endswith(".bias") or "layernorm" in name.lower() or "layer_norm" in name.lower() or "norm.weight" in name.lower()


def build_optimizer(model, log_tau, cfg):
    backbone_names = ("vivit", "wavlm")

    bb_decay, bb_no_decay = [], []
    hd_decay, hd_no_decay = [], []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_backbone = any(name.startswith(b) for b in backbone_names)
        no_decay = is_no_decay_param(name)
        if is_backbone:
            (bb_no_decay if no_decay else bb_decay).append(p)
        else:
            (hd_no_decay if no_decay else hd_decay).append(p)

    param_groups = [
        {"params": bb_decay,    "lr": cfg["lr_backbone"], "weight_decay": cfg["weight_decay"], "name": "backbone_decay"},
        {"params": bb_no_decay, "lr": cfg["lr_backbone"], "weight_decay": 0.0,                 "name": "backbone_no_decay"},
        {"params": hd_decay,    "lr": cfg["lr_head"],     "weight_decay": cfg["weight_decay"], "name": "head_decay"},
        {"params": hd_no_decay, "lr": cfg["lr_head"],     "weight_decay": 0.0,                 "name": "head_no_decay"},
        {"params": [log_tau],   "lr": cfg["lr_head"],     "weight_decay": 0.0,                 "name": "log_tau"},
    ]

    # Sanity print
    for g in param_groups:
        n_params = sum(p.numel() for p in g["params"])
        print(f"  {g['name']:<22}  lr={g['lr']:<7.0e}  wd={g['weight_decay']:.3f}  params={n_params/1e6:.2f}M")

    return torch.optim.AdamW(param_groups)


print("Building optimizer...")
optimizer = build_optimizer(model, log_tau, CFG)

## Phase E — Cosine schedule with linear warmup

Linear from 0 → 1 over `warmup_steps`, then cosine decay to 0 over the remaining steps. Standard schedule for fine-tuning large transformers.

Number of optimizer steps per epoch = `ceil(len(train_ds) / (micro_batch × grad_accum))`.

In [ ]:
def build_scheduler(optimizer, num_training_steps: int, warmup_steps: int):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        # Cosine decay from 1.0 to 0.0
        progress = (step - warmup_steps) / max(1, num_training_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Compute step counts
steps_per_epoch  = math.ceil(len(train_ds) / (CFG["micro_batch"] * CFG["grad_accum"]))
total_steps      = steps_per_epoch * CFG["max_epochs"]
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps    : {total_steps}")
print(f"Warmup steps   : {CFG['warmup_steps']}")

scheduler = build_scheduler(optimizer, total_steps, CFG["warmup_steps"])

## Phase F — `train_one_epoch` and `validate` functions

**Training step logic:**
- Move batch to GPU
- 50% of batches: apply audio mixup; loss is mixed CE on the four 4-class outputs
- 50% of batches: clean forward; loss is standard CE with label smoothing
- Contrastive loss always applied (uses non-mixed video + non-mixed audio relationships indirectly through the embeddings)
- Backward inside `autocast(bf16)`
- Gradient accumulation over `grad_accum` micro-batches before optimizer step
- Grad clip at max-norm 1.0

**Validation:**
- No mixup, no autocast (fp32 for precision in metrics)
- Collect predictions and labels across all val batches
- Compute: 4-class balanced accuracy, AUC (macro OvR), macro F1, binary balanced accuracy
- Return dict

In [ ]:
def compute_loss(out, batch, log_tau, cfg, mixup_info=None):
    """Compute total loss for one micro-batch.

    mixup_info: None for clean batch, or (labels_b, lam) for mixup batch.
    """
    labels_a = batch["label_4class"]
    aux_w  = cfg["aux_loss_weight"]
    cons_w = cfg["contrastive_weight"]
    ls     = cfg["label_smoothing"]

    # --- Detection losses ---
    if mixup_info is None:
        # Clean batch
        # Ensemble logits are already log-softmax → use NLL
        l_main = F.nll_loss(out["logits_4class"], labels_a)
        l_v = F.cross_entropy(out["logits_video"], labels_a, label_smoothing=ls)
        l_a = F.cross_entropy(out["logits_audio"], labels_a, label_smoothing=ls)
        l_f = F.cross_entropy(out["logits_fused"], labels_a, label_smoothing=ls)
    else:
        labels_b, lam = mixup_info
        # NLL doesn't accept label_smoothing — apply mixed NLL manually
        log_probs = out["logits_4class"]
        l_main_a = F.nll_loss(log_probs, labels_a)
        l_main_b = F.nll_loss(log_probs, labels_b)
        l_main = lam * l_main_a + (1 - lam) * l_main_b
        l_v = mixed_ce(out["logits_video"], labels_a, labels_b, lam, ls)
        l_a = mixed_ce(out["logits_audio"], labels_a, labels_b, lam, ls)
        l_f = mixed_ce(out["logits_fused"], labels_a, labels_b, lam, ls)

    l_det = l_main + aux_w * (l_v + l_a + l_f) / 3.0

    # --- Contrastive loss with learnable temperature ---
    tau_clamped = log_tau.clamp(log_tau_min, log_tau_max).exp()
    v_emb = out["v_embed"]; a_emb = out["a_embed"]
    B = v_emb.size(0)
    sim = v_emb @ a_emb.T / tau_clamped
    targets = torch.arange(B, device=sim.device)
    l_cons = 0.5 * (F.cross_entropy(sim, targets) + F.cross_entropy(sim.T, targets))

    total = l_det + cons_w * l_cons
    return {"total": total, "det": l_det.detach(), "cons": l_cons.detach(), "main": l_main.detach()}


def train_one_epoch(model, loader, optimizer, scheduler, log_tau, cfg, epoch_idx, log_file=None):
    model.train()
    epoch_losses = defaultdict(list)
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc=f"Epoch {epoch_idx+1}/{cfg['max_epochs']} [train]")
    accum_count = 0
    step_in_epoch = 0

    for batch_idx, batch in enumerate(pbar):
        faces = batch["faces"].to(device, non_blocking=True)
        audio = batch["audio"].to(device, non_blocking=True)
        labels = batch["label_4class"].to(device, non_blocking=True)

        # Decide mixup
        use_mixup = (random.random() < cfg["audio_mixup_prob"])
        if use_mixup:
            audio_mixed, lab_a, lab_b, lam = audio_mixup(audio, labels, cfg["audio_mixup_alpha"])
            mixup_info = (lab_b, lam)
            audio_in = audio_mixed
        else:
            mixup_info = None
            audio_in = audio

        # Forward + loss in bf16 autocast
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = model(faces, audio_in)
            losses = compute_loss(out, {"label_4class": lab_a if use_mixup else labels}, log_tau, cfg, mixup_info)
            loss_scaled = losses["total"] / cfg["grad_accum"]

        loss_scaled.backward()
        accum_count += 1

        # Optimizer step every grad_accum micro-batches
        if accum_count == cfg["grad_accum"]:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad] + [log_tau],
                max_norm=cfg["grad_clip"],
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            accum_count = 0
            step_in_epoch += 1

        # Track losses
        for k in ("total", "det", "cons"):
            epoch_losses[k].append(float(losses[k]))

        # Update progress bar
        if batch_idx % 10 == 0:
            pbar.set_postfix(
                loss=f"{losses['total']:.3f}",
                det=f"{losses['det']:.3f}",
                cons=f"{losses['cons']:.3f}",
                tau=f"{current_tau():.4f}",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            )

    return {k: float(np.mean(v)) for k, v in epoch_losses.items()}


@torch.no_grad()
def validate(model, loader, log_tau, cfg, epoch_idx):
    model.eval()
    all_logits, all_labels_4c, all_labels_bin = [], [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch_idx+1}/{cfg['max_epochs']} [val]  ", leave=False)
    for batch in pbar:
        faces  = batch["faces"].to(device, non_blocking=True)
        audio  = batch["audio"].to(device, non_blocking=True)
        out    = model(faces, audio)
        all_logits.append(out["logits_4class"].float().cpu())   # log-softmax outputs
        all_labels_4c.append(batch["label_4class"])
        all_labels_bin.append(batch["label_binary"])

    logits     = torch.cat(all_logits, dim=0)
    labels_4c  = torch.cat(all_labels_4c, dim=0).numpy()
    labels_bin = torch.cat(all_labels_bin, dim=0).numpy()
    probs_4c   = logits.exp().numpy()
    preds_4c   = probs_4c.argmax(axis=1)

    # 4-class metrics
    acc_4c = (preds_4c == labels_4c).mean()
    bacc_4c = balanced_accuracy_score(labels_4c, preds_4c)
    try:
        auc_4c = roc_auc_score(labels_4c, probs_4c, multi_class="ovr", average="macro")
    except Exception:
        auc_4c = float("nan")
    f1_macro = f1_score(labels_4c, preds_4c, average="macro")

    # Binary metrics (treat 1/2/3 as fake, 0 as real)
    preds_bin = (preds_4c > 0).astype(int)
    probs_fake = 1 - probs_4c[:, 0]   # P(any fake)
    bacc_bin = balanced_accuracy_score(labels_bin, preds_bin)
    try:
        auc_bin = roc_auc_score(labels_bin, probs_fake)
    except Exception:
        auc_bin = float("nan")

    return {
        "acc_4c": float(acc_4c), "bacc_4c": float(bacc_4c), "auc_4c": float(auc_4c), "f1_macro": float(f1_macro),
        "bacc_bin": float(bacc_bin), "auc_bin": float(auc_bin),
    }

## Phase G — Smoke test on tiny subset (1 epoch)

Run the full training pipeline on 50 train samples + 20 val samples for one epoch. Catches every bug — data loading, mixup math, autocast issues, optimizer steps, validation, scheduler stepping — in **~3 minutes** instead of 45.

If this passes, the full run will work.

In [ ]:
# --- Build tiny datasets ---
class SubsetDataset(Dataset):
    def __init__(self, base, indices):
        self.base = base
        self.indices = list(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]

random.seed(SEED)
train_indices = random.sample(range(len(train_ds)), 50)
val_indices   = random.sample(range(len(val_ds)),   20)
tiny_train = SubsetDataset(train_ds, train_indices)
tiny_val   = SubsetDataset(val_ds,   val_indices)

tiny_train_loader = DataLoader(tiny_train, batch_size=CFG["micro_batch"], shuffle=True,
                                num_workers=0, collate_fn=collate_fn, pin_memory=True)
tiny_val_loader   = DataLoader(tiny_val,   batch_size=CFG["micro_batch"], shuffle=False,
                                num_workers=0, collate_fn=collate_fn, pin_memory=True)

# --- Build fresh model + optimizer for smoke test (avoid mutating the real ones) ---
print("\n=== SMOKE TEST: 1 epoch on 50 train + 20 val samples ===\n")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()
train_metrics = train_one_epoch(model, tiny_train_loader, optimizer, scheduler, log_tau, CFG, epoch_idx=0)
val_metrics   = validate(model, tiny_val_loader, log_tau, CFG, epoch_idx=0)
elapsed = time.time() - t0

print(f"\nSmoke test complete in {elapsed:.1f}s")
print(f"\nTrain losses:")
for k, v in train_metrics.items(): print(f"  {k:<6}: {v:.4f}")
print(f"\nVal metrics:")
for k, v in val_metrics.items(): print(f"  {k:<10}: {v:.4f}")

peak_vram = torch.cuda.max_memory_allocated() / 1e9
print(f"\nPeak VRAM during smoke test: {peak_vram:.2f} GB")
print("If this looks healthy (loss is finite, val metrics are between 0 and 1), proceed to Phase H.")

## Phase H — Full training run

Builds the real model fresh (resets the smoke-test state), then trains for up to `max_epochs` with early stopping. Saves:

- `checkpoints/best.pt` — best by val_bacc_4c (loaded by nb06 for final evaluation)
- `checkpoints/last.pt` — every epoch (for resumption)
- `results/tables/training_log.csv` — per-epoch metrics
- `results/logs/train_TIMESTAMP.log` — stdout

**Resume:** if `checkpoints/last.pt` exists, training resumes from that epoch automatically. To start fresh, delete `last.pt`.

**~6 hours.** Don't close the kernel or AnyDesk session.

In [ ]:
# --- Build REAL data loaders ---
train_loader = DataLoader(train_ds, batch_size=CFG["micro_batch"], shuffle=True,
                          num_workers=CFG["num_workers"], collate_fn=collate_fn,
                          pin_memory=True, persistent_workers=(CFG["num_workers"] > 0))
val_loader   = DataLoader(val_ds,   batch_size=CFG["micro_batch"], shuffle=False,
                          num_workers=CFG["num_workers"], collate_fn=collate_fn,
                          pin_memory=True, persistent_workers=(CFG["num_workers"] > 0))

# --- Reset model + optimizer for fresh run ---
print("Rebuilding model + optimizer for fresh training run...")
torch.cuda.empty_cache()

del model, optimizer, scheduler, log_tau
torch.cuda.empty_cache()

model = AVXAIModel(use_gradient_checkpointing=True).to(device)
log_tau = nn.Parameter(torch.log(torch.tensor(CFG["tau_init"])).to(device))
optimizer = build_optimizer(model, log_tau, CFG)

steps_per_epoch = math.ceil(len(train_ds) / (CFG["micro_batch"] * CFG["grad_accum"]))
total_steps     = steps_per_epoch * CFG["max_epochs"]
scheduler = build_scheduler(optimizer, total_steps, CFG["warmup_steps"])

# --- Resume check ---
LAST_CKPT = CKPT_DIR / "last.pt"
BEST_CKPT = CKPT_DIR / "best.pt"
start_epoch = 0
best_bacc = -float("inf")
history = []
epochs_since_improvement = 0

if LAST_CKPT.exists():
    print(f"\nFound existing checkpoint at {LAST_CKPT}. Resuming...")
    ck = torch.load(LAST_CKPT, weights_only=False, map_location=device)
    model.load_state_dict(ck["model_state"])
    optimizer.load_state_dict(ck["optimizer_state"])
    scheduler.load_state_dict(ck["scheduler_state"])
    log_tau.data = ck["log_tau"].to(device)
    start_epoch = ck["epoch"] + 1
    best_bacc   = ck["best_bacc"]
    history     = ck["history"]
    epochs_since_improvement = ck.get("epochs_since_improvement", 0)
    print(f"Resuming from epoch {start_epoch}; best_bacc so far: {best_bacc:.4f}")
else:
    print("No checkpoint found. Starting fresh.")

# --- Set up logging ---
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_fp = LOGS_DIR / f"train_{ts}.log"
training_log_csv = TABLES_DIR / "training_log.csv"
write_csv_header = not training_log_csv.exists() or start_epoch == 0

# --- Train loop ---
print(f"\n{'='*70}\nStarting training: epochs {start_epoch} → {CFG['max_epochs']-1}\n{'='*70}\n")
training_start = time.time()

for epoch in range(start_epoch, CFG["max_epochs"]):
    epoch_start = time.time()
    print(f"\n--- Epoch {epoch+1}/{CFG['max_epochs']} ---")

    # Train
    torch.cuda.reset_peak_memory_stats()
    train_metrics = train_one_epoch(model, train_loader, optimizer, scheduler, log_tau, CFG, epoch)
    peak_vram = torch.cuda.max_memory_allocated() / 1e9

    # Validate
    val_metrics = validate(model, val_loader, log_tau, CFG, epoch)

    epoch_time = time.time() - epoch_start
    epoch_record = {
        "epoch":       epoch + 1,
        "train_total": train_metrics["total"],
        "train_det":   train_metrics["det"],
        "train_cons":  train_metrics["cons"],
        "val_acc_4c":  val_metrics["acc_4c"],
        "val_bacc_4c": val_metrics["bacc_4c"],
        "val_auc_4c":  val_metrics["auc_4c"],
        "val_f1_macro": val_metrics["f1_macro"],
        "val_bacc_bin": val_metrics["bacc_bin"],
        "val_auc_bin": val_metrics["auc_bin"],
        "tau":         current_tau(),
        "lr_backbone": optimizer.param_groups[0]["lr"],
        "lr_head":     optimizer.param_groups[2]["lr"],
        "epoch_time":  epoch_time,
        "peak_vram":   peak_vram,
    }
    history.append(epoch_record)

    # Print epoch summary
    print(f"  Train loss      : {train_metrics['total']:.4f}  (det={train_metrics['det']:.4f}, cons={train_metrics['cons']:.4f})")
    print(f"  Val BAcc (4-cls): {val_metrics['bacc_4c']:.4f}")
    print(f"  Val AUC  (4-cls): {val_metrics['auc_4c']:.4f}")
    print(f"  Val F1 macro    : {val_metrics['f1_macro']:.4f}")
    print(f"  Val BAcc (bin)  : {val_metrics['bacc_bin']:.4f}")
    print(f"  τ (learned)     : {current_tau():.4f}")
    print(f"  Peak VRAM       : {peak_vram:.2f} GB")
    print(f"  Epoch time      : {epoch_time/60:.1f} min")

    # Save training_log.csv
    with open(training_log_csv, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(epoch_record.keys()))
        if write_csv_header:
            writer.writeheader()
            write_csv_header = False
        writer.writerow(epoch_record)

    # Always save "last.pt"
    ck_payload = {
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "log_tau":         log_tau.detach().cpu(),
        "best_bacc":       best_bacc,
        "history":         history,
        "epochs_since_improvement": epochs_since_improvement,
        "cfg":             CFG,
    }
    torch.save(ck_payload, LAST_CKPT)

    # Save "best.pt" if improved
    improved = val_metrics["bacc_4c"] > best_bacc
    if improved:
        best_bacc = val_metrics["bacc_4c"]
        epochs_since_improvement = 0
        ck_payload["best_bacc"] = best_bacc
        torch.save(ck_payload, BEST_CKPT)
        print(f"  ** NEW BEST val BAcc: {best_bacc:.4f}. Saved to {BEST_CKPT.name}")
    else:
        epochs_since_improvement += 1
        print(f"  No improvement for {epochs_since_improvement} epochs (best={best_bacc:.4f}, patience={CFG['patience']})")
        if epochs_since_improvement >= CFG["patience"]:
            print(f"\nEarly stopping triggered after epoch {epoch+1}.")
            break

total_time = time.time() - training_start
print(f"\n{'='*70}")
print(f"Training complete in {total_time/3600:.2f} hours")
print(f"Best val BAcc (4-class): {best_bacc:.4f}")
print(f"Best checkpoint: {BEST_CKPT}")
print(f"{'='*70}")

## Phase I — Save final summary

Write a small summary JSON next to the checkpoint with the final metrics and config, so nb06 can pick it up without re-parsing.

In [ ]:
summary = {
    "best_epoch":      max(history, key=lambda r: r["val_bacc_4c"])["epoch"] if history else None,
    "best_val_bacc_4c": best_bacc,
    "total_epochs_run": len(history),
    "total_training_hours": total_time / 3600,
    "final_tau":       current_tau(),
    "config":          CFG,
}

summary_fp = CKPT_DIR / "training_summary.json"
with open(summary_fp, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {summary_fp}")
print("\nSummary:")
print(json.dumps(summary, indent=2))

## Done.

**What's on disk now:**
```
checkpoints/
├── best.pt                ← LOAD THIS in nb06 for final evaluation
├── last.pt                ← latest epoch (resume-from)
└── training_summary.json  ← config + best metrics

results/
├── tables/training_log.csv  ← per-epoch CSV (for plotting in nb06)
└── logs/train_*.log         ← stdout
```

**Git commit:**
```cmd
git add notebooks/04_training.ipynb results/tables/training_log.csv checkpoints/training_summary.json
git commit -m "nb04: training run complete — best val BAcc=X.XXXX, X epochs"
git push origin main
```

⚠️ **Do NOT commit `checkpoints/best.pt` or `last.pt`** — they're 1.6 GB each. Should already be covered by the `checkpoints/` rule in `.gitignore`; add it if not.

**What's next: nb06 (evaluation).**
Loads `best.pt`, runs test-set inference, generates all 7 tables + 5 figures (training curves, confusion matrices, ROC, modality ablation). nb05 (adversarial ensemble) is skipped per project decision — we go straight from base training to evaluation + XAI.